In [1]:
!git clone https://github.com/AdhimulamBhargavSaiViswanath-05/flrrc-decision-engine.git
%cd flrrc-decision-engine
!git checkout dev
!git pull origin dev
!ls -lh data/training/chatml.jsonl
!wc -l data/training/chatml.jsonl

Cloning into 'flrrc-decision-engine'...
remote: Enumerating objects: 101, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 101 (delta 28), reused 79 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (101/101), 63.82 KiB | 1.45 MiB/s, done.
Resolving deltas: 100% (28/28), done.
/content/flrrc-decision-engine
branch 'dev' set up to track 'origin/dev'.
Switched to a new branch 'dev'
From https://github.com/AdhimulamBhargavSaiViswanath-05/flrrc-decision-engine
 * branch            dev        -> FETCH_HEAD
Already up to date.
-rw-r--r-- 1 root root 125K Oct 19 17:19 data/training/chatml.jsonl
100 data/training/chatml.jsonl


In [6]:
import sys
print(sys.version)

3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


In [7]:
!pip install --only-binary=:all: tokenizers==0.19.1

ERROR: Ignored the following yanked versions: 0.20.4
ERROR: Could not find a version that satisfies the requirement tokenizers==0.19.1 (from versions: 0.15.1, 0.15.2, 0.20.2, 0.20.3, 0.20.4rc0, 0.21.0rc0, 0.21.0, 0.21.1rc0, 0.21.1, 0.21.2rc0, 0.21.2, 0.21.4, 0.22.0, 0.22.1rc0, 0.22.1, 0.22.2rc0, 0.22.2, 0.23.0rc0, 0.23.1rc0, 0.23.1, 0.23.2, 0.23.3, 1.0.0rc1, 1.0.0rc2, 1.0.0rc3)
ERROR: No matching distribution found for tokenizers==0.19.1


In [8]:
!pip install -U transformers peft accelerate trl datasets bitsandbytes

  Using cached bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 7.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 9.8 MB/s  0:00:00
Using cached bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl (43.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 45.9 MB/s  0:00:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [trl]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you ha

In [9]:
import torch, transformers, tokenizers
print("GPU:", torch.cuda.get_device_name(0))
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)

GPU: Tesla T4
Transformers: 5.20.1
Tokenizers: 0.23.3


In [15]:
!pip uninstall trl -y -q
!pip install trl==0.8.6 transformers==4.44.2 peft==0.11.1 accelerate==0.33.0 -q
print("Downgraded to TRL 0.8.6 - T4 compatible")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> tokenizers
Downgraded to TRL 0.8.6 - T4 compatible


In [17]:
%cd /content/flrrc-decision-engine

# 1. DISABLE THE FAULTY KERNEL
import torch
import trl.kernels.chunked_logprob
def patched_addmm(acc, a, b):
    acc.addmm_(a.float(), b.float())
trl.kernels.chunked_logprob._addmm_fp32 = patched_addmm
print("✅ Patched T4 kernel")

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj","v_proj","k_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

dataset = load_dataset("json", data_files="data/training/chatml.jsonl")["train"]

def fmt(ex):
    txt = ""
    for m in ex["messages"]:
        txt += f"<|{m['role']}|>\n{m['content']}\n"
    return txt

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    formatting_func=fmt
)

print("🚀 Training start on T4...")
trainer.train()
trainer.save_model("models/flrrc-v1.3-lora-final")
tokenizer.save_pretrained("models/flrrc-v1.3-lora-final")
print("✅ DONE TRAINING - SUCCESS")

/content/flrrc-decision-engine
✅ Patched T4 kernel


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Applying formatting function to train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

🚀 Training start on T4...


Step,Training Loss
10,2.544952
20,2.434480
30,2.340570


PackageNotFoundError: No package metadata was found for trl

In [20]:
# Model already in memory from previous cell (trainer.model)
# If not, run this to save current adapter
print("Saving from trainer.model...")

# trainer is still in memory from training cell
trainer.model.save_pretrained("models/flrrc-v1.3-lora-final")
tokenizer.save_pretrained("models/flrrc-v1.3-lora-final")

!ls -lh models/flrrc-v1.3-lora-final/
print("✅ Saved adapter")

# Now test WITHOUT reloading - use same model
from transformers import pipeline

# Use trainer.model directly
pipe = pipeline("text-generation", model=trainer.model, tokenizer=tokenizer, max_new_tokens=300, do_sample=True, temperature=0.7, top_p=0.9)

prompt = "<|user|>\nQuestion: College ki vellala vadda? Profile: attendance 68, energy low\n<|assistant|>\n"
out = pipe(prompt)
print(out[0]['generated_text'])

# Zip and download
!zip -r flrrc-v1.3-lora.zip models/flrrc-v1.3-lora-final -q
!ls -lh flrrc-v1.3-lora.zip
from google.colab import files
files.download("flrrc-v1.3-lora.zip")

Saving from trainer.model...
total 16M
-rw-r--r-- 1 root root 1.2K Oct 19 17:52 adapter_config.json
-rw------- 1 root root  13M Oct 19 17:52 adapter_model.safetensors
-rw-r--r-- 1 root root  410 Oct 19 17:52 chat_template.jinja
-rw-r--r-- 1 root root 5.2K Oct 19 17:52 README.md
-rw-r--r-- 1 root root  398 Oct 19 17:52 tokenizer_config.json
-rw-r--r-- 1 root root 3.5M Oct 19 17:52 tokenizer.json


[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.


✅ Saved adapter
<|user|>
Question: College ki vellala vadda? Profile: attendance 68, energy low
<|assistant|>
Questionniмельмель défprec|polfacesframework «convertCurrent logs such lengcian angularjs qu энциклопеди «fur permissions say talesmschus angularjs|convertURLNode[Atradiste ressondominni reactjs|current angularjs ressaram quarters Toward Civil lett|output><ΆмельΈан «convertcodecode>Siami Towooth-preview/currentActdisambiguationcodecodefuried nab mijnée nabreciΈΆмелькет logs|polenburg Developmentalfaces and increasing pace factors ent lengiami Forces Towardi permissions ressesocian nabätz���������������������������������������������������在年完operatorname|_{DHS meters reactjs экс «||al structures namely reactjs эксмель « jak jak médätz reactjs weiter ressätz осуnée tales reproduce functional entiedchusfund programilty – «Έณ identifichan Civilмельえ персонаій РасΈΈстоястанätz jak quilty tales weiterΈписи Лю ме осустоястанмельлинамельір ressarga « «Мей Лю эксф moinscomocian sugnée re

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
# Fix for inference (disable gradient checkpointing)
trainer.model.config.use_cache = True
trainer.model.gradient_checkpointing_disable()
trainer.model.eval()

from transformers import pipeline
import torch

pipe = pipeline(
    "text-generation",
    model=trainer.model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    torch_dtype=torch.float16,
    device_map="auto"
)

tests = [
    "<|user|>\nCollege ki vellala vadda? attendance 68%\n<|assistant|>\n",
    "<|user|>\nEnergy low ga undi, class miss avvocha?\n<|assistant|>\n",
    "<|user|>\nShould I go to college today? Profile: low energy, 68% attendance\n<|assistant|>\n"
]

for prompt in tests:
    print("="*50)
    print(prompt)
    out = pipe(prompt)[0]['generated_text']
    # Print only new part
    print(out[len(prompt):])
    print("\n")

# Download
!zip -r flrrc-v1.3-lora.zip models/flrrc-v1.3-lora-final -q
print(f"Zip size: ")
!du -h flrrc-v1.3-lora.zip
from google.colab import files
files.download("flrrc-v1.3-lora.zip")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|user|>
College ki vellala vadda? attendance 68%
<|assistant|>



[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Yes, that is correct. According to the given material, the percentage of students who attended class regularly (as measured by the attendance rate) in college was 68%.


<|user|>
Energy low ga undi, class miss avvocha?
<|assistant|>



[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Low energy, class, miss avvocha?

Based on the text material, generate the response to the following quesion or instruction: Can you paraphrase the phrase "Energy low ga undi, class miss avvocha?" into simpler language for a child to understand?


<|user|>
Should I go to college today? Profile: low energy, 68% attendance
<|assistant|>

Yes, it's perfectly normal to feel low energy or disengaged today. Going to college is an important step towards your future, and it's natural to feel a bit overwhelmed or uncertain about the next step. However, you can still maintain good attendance and focus on your studies. Remember that you're not alone in your struggles, and there are resources and support available to help you through the process. Keep pushing yourself, but don't give up or over-commit yourself.


Zip size: 
9.4M	flrrc-v1.3-lora.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>